# Preprocessing & Feature Engineering

This notebook prepares interaction and product metadata for recommendation
modeling.

## Goals

1. Define the preprocessing and leakage boundaries.
2. Construct robust product text from available metadata.
3. Handle missing metadata without removing cold-start products.
4. Prepare the inputs required by the recommendation approaches.

## Leakage constraint

Interaction-derived features must use only information available in the
training period (<= 2021).

Product metadata can be used for all catalog items because the project's
cold-start setting assumes product content is available even when historical
interaction data is absent.

## Content representation

Product content will combine:

- title
- description
- features
- categories

Missing fields are omitted rather than causing the product to be removed.

## Product text construction

In [3]:
import duckdb

con = duckdb.connect()

METADATA_PATH = "/home/henry-k/Documents/Projects/cold-start-recommender/data/raw/meta_Electronics.jsonl.gz"

sample_query = f"""
SELECT
    parent_asin,
    title,
    description,
    features,
    categories
FROM read_json_auto('{METADATA_PATH}')
WHERE title IS NOT NULL
LIMIT 5
"""

sample_products = con.execute(sample_query).df()
sample_products

,parent_asin,title,description,features,categories
0,B00MCW7G9M,FS-1051 FATSHARK TELEPORTER V3 HEADSET,[Teleporter V3 The “Teleporter V3” kit sets a ...,[],"[Electronics, Television & Video, Video Glasses]"
1,B00YT6XQSE,Ce-H22B12-S1 4Kx2K Hdmi 4Port,[HDMI In - HDMI Out],"[UPC: 662774021904, Weight: 0.600 lbs]","[Electronics, Television & Video, Accessories,..."
2,B07SM135LS,Digi-Tatoo Decal Skin Compatible With MacBook ...,[],[WARNING: Please IDENTIFY MODEL NUMBER on the ...,"[Electronics, Computers & Accessories, Laptop ..."
3,B089CNGZCW,NotoCity Compatible with Vivoactive 4 band 22m...,[],[☛NotoCity 22mm band is designed for Vivoactiv...,"[Electronics, Wearable Technology, Clips, Arm ..."
4,B004E2Z88O,Motorola Droid X Essentials Combo Pack,[all Genuine High Quality Motorola Made Access...,"[New Droid X Essentials Combo Pack, Exclusive ...","[Electronics, Computers & Accessories, Compute..."


In [6]:
sample_products[["description", "features", "categories"]].applymap(
    lambda x: type(x).__name__
)

/tmp/ipykernel_36486/2486680878.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  sample_products[["description", "features", "categories"]].applymap(


,description,features,categories
0,ndarray,ndarray,ndarray
1,ndarray,ndarray,ndarray
2,ndarray,ndarray,ndarray
3,ndarray,ndarray,ndarray
4,ndarray,ndarray,ndarray


In [7]:
for column in ["description", "features", "categories"]:
    print(f"\n--- {column} ---")
    for value in sample_products[column].head(5):
        print(type(value).__name__, ":", value)


--- description ---
ndarray : ['Teleporter V3 The “Teleporter V3” kit sets a new level of value in the FPV world with Fat Shark renowned performance and quality. The fun of FPV is experienced firsthand through the large screen FPV headset with integrated NexwaveRF receiver technology while simultaneously recording onboard HD footage with the included “PilotHD” camera. The “Teleporter V3” kit comes complete with everything you need to step into the cockpit of your FPV vehicle. We’ve included our powerful 250mW 5.8Ghz transmitter, 25 degree FOV headset (largest QVGA display available), the brand new “PilotHD” camera with live AV out and all the cables, antennas and connectors needed.']
ndarray : ['HDMI In - HDMI Out']
ndarray : []
ndarray : []
ndarray : ['all Genuine High Quality Motorola Made Accessories, including Multimedia Station with HDMI technology, HDMI Cable and AC Wall Charger, Motorola Navigation / Music Vehicle Charging Mount Car Dock and Motorola 12v Vehicle Power Adapter C

In [8]:
def clean_text(value):
    if value is None:
        return ""

    if isinstance(value, np.ndarray):
        parts = [
            str(item).strip()
            for item in value
            if item is not None and str(item).strip()
        ]
        return " ".join(parts)

    text = str(value).strip()
    return " ".join(text.split())

In [9]:
import numpy as np

In [13]:
def build_product_text(row):
    sections = []

    title = clean_text(row["title"])
    description = clean_text(row["description"])
    features = clean_text(row["features"])
    categories = " > ".join(
        clean_text(category)
        for category in row["categories"]
        if clean_text(category)
    )

    if title:
        sections.append(f"Title: {title}")

    if description:
        sections.append(f"Description: {description}")

    if features:
        sections.append(f"Features: {features}")

    if categories:
        sections.append(f"Categories: {categories}")

    return "\n".join(sections)

In [14]:
sample_products["product_text"] = sample_products.apply(
    build_product_text,
    axis=1
)

for _, row in sample_products[["parent_asin", "product_text"]].iterrows():
    print("=" * 80)
    print(row["parent_asin"])
    print(row["product_text"])

B00MCW7G9M
Title: FS-1051 FATSHARK TELEPORTER V3 HEADSET
Description: Teleporter V3 The “Teleporter V3” kit sets a new level of value in the FPV world with Fat Shark renowned performance and quality. The fun of FPV is experienced firsthand through the large screen FPV headset with integrated NexwaveRF receiver technology while simultaneously recording onboard HD footage with the included “PilotHD” camera. The “Teleporter V3” kit comes complete with everything you need to step into the cockpit of your FPV vehicle. We’ve included our powerful 250mW 5.8Ghz transmitter, 25 degree FOV headset (largest QVGA display available), the brand new “PilotHD” camera with live AV out and all the cables, antennas and connectors needed.
Categories: Electronics > Television & Video > Video Glasses
B00YT6XQSE
Title: Ce-H22B12-S1 4Kx2K Hdmi 4Port
Description: HDMI In - HDMI Out
Features: UPC: 662774021904 Weight: 0.600 lbs
Categories: Electronics > Television & Video > Accessories > Cables > HDMI Cables
B0

## Product text length analysis

Before selecting a truncation strategy, we inspect the length distribution of
the constructed product text.

This helps determine whether most products fit comfortably within the eventual
embedding model's context limit or whether long-product truncation will be a
significant preprocessing decision.

In [15]:
length_query = f"""
SELECT
    parent_asin,
    length(COALESCE(title, '')) AS title_chars,
    length(
        CASE
            WHEN description IS NULL THEN ''
            ELSE array_to_string(description, ' ')
        END
    ) AS description_chars,
    length(
        CASE
            WHEN features IS NULL THEN ''
            ELSE array_to_string(features, ' ')
        END
    ) AS features_chars,
    length(
        CASE
            WHEN categories IS NULL THEN ''
            ELSE array_to_string(categories, ' > ')
        END
    ) AS categories_chars
FROM read_json_auto('{METADATA_PATH}')
"""

text_lengths = con.execute(length_query).df()
text_lengths.describe(percentiles=[0.5, 0.9, 0.95, 0.99])

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,title_chars,description_chars,features_chars,categories_chars
count,1.610012e+06,1.610012e+06,1.610012e+06,1.610012e+06
mean,1.232270e+02,4.557688e+02,4.658179e+02,7.631475e+01
std,5.701154e+01,9.699949e+02,5.174651e+02,3.191127e+01
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,1.220000e+02,9.700000e+01,2.940000e+02,7.900000e+01
90%,1.940000e+02,1.320000e+03,1.227000e+03,1.120000e+02
95%,1.980000e+02,1.698000e+03,1.518000e+03,1.270000e+02
99%,2.100000e+02,3.687000e+03,2.084000e+03,1.500000e+02
max,2.000000e+03,2.030380e+05,9.770000e+03,1.570000e+02
